<a href="https://colab.research.google.com/github/tarun6762/The-Data-And-AI-Developers-project/blob/main/Redrob.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install pandas sentence-transformers scikit-learn


In [ ]:
from __future__ import annotations

import csv
import logging
import re
import sys
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
from sentence_transformers import CrossEncoder, SentenceTransformer, util
from sklearn.preprocessing import MinMaxScaler

# ─────────────────────────────────────────────
# Logging
# ─────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("CDE")


# ─────────────────────────────────────────────
# Configuration (single source of truth)
# ─────────────────────────────────────────────
@dataclass
class EngineConfig:
    """All tunable knobs live here. No magic numbers in logic."""

    # Models
    bi_encoder: str = "all-mpnet-base-v2"          # fast dense retrieval
    cross_encoder: str = "cross-encoder/ms-marco-MiniLM-L-6-v2"  # precision reranking

    # Composite score weights (must sum to 1.0)
    w_semantic: float  = 0.50   # contextual fit via bi-encoder cosine sim
    w_rerank:   float  = 0.20   # cross-encoder rerank score
    w_exp:      float  = 0.15   # log-scaled years of experience
    w_skill:    float  = 0.10   # weighted skill density
    w_seniority: float = 0.05   # title seniority signal

    # Only rerank top-K to keep cross-encoder cost manageable
    rerank_top_k: int = 50

    # Skill taxonomy  {skill_term: weight}
    # Tier-1 = 3.0 (must-haves), Tier-2 = 2.0 (strong), Tier-3 = 1.0 (bonus)
    skill_taxonomy: dict[str, float] = field(default_factory=lambda: {
        # Tier-1 – must-haves for most data/AI roles
        "python":           3.0,
        "sql":              3.0,
        "machine learning": 3.0,
        "ml":               3.0,
        "data pipeline":    3.0,
        "spark":            3.0,
        # Tier-2 – strong positives
        "llm":              2.0,
        "genai":            2.0,
        "deep learning":    2.0,
        "nlp":              2.0,
        "data science":     2.0,
        "neural network":   2.0,
        # Tier-3 – nice-to-have / context
        "ai":               1.0,
        "chatgpt":          1.0,
        "computer vision":  1.0,
        "kubernetes":       1.0,
        "docker":           1.0,
        "airflow":          1.0,
        "dbt":              1.0,
        "mlflow":           1.0,
    })

    # Seniority keyword → score (0–1)
    seniority_map: dict[str, float] = field(default_factory=lambda: {
        "staff":      1.00,
        "principal":  1.00,
        "vp":         0.95,
        "head of":    0.95,
        "director":   0.90,
        "lead":       0.85,
        "senior":     0.80,
        "sr.":        0.80,
        "sr ":        0.80,
        "mid":        0.60,
        "junior":     0.35,
        "jr.":        0.35,
        "intern":     0.10,
        "trainee":    0.10,
    })

    def __post_init__(self):
        total = self.w_semantic + self.w_rerank + self.w_exp + self.w_skill + self.w_seniority
        if not abs(total - 1.0) < 1e-6:
            raise ValueError(f"Weights must sum to 1.0, got {total:.4f}")


# ─────────────────────────────────────────────
# Typed result
# ─────────────────────────────────────────────
@dataclass
class ScoredCandidate:
    candidate_id: str
    rank: int
    score: float
    semantic_score: float
    rerank_score: float
    exp_score: float
    skill_score: float
    seniority_score: float
    reasoning: str


# ─────────────────────────────────────────────
# Engine
# ─────────────────────────────────────────────
class CandidateDiscoveryEngine:
    """
    Pipeline:
      load → preprocess → bi-encode → top-K rerank → composite score → explain
    """

    def __init__(self, config: Optional[EngineConfig] = None):
        self.cfg = config or EngineConfig()
        self._scaler = MinMaxScaler()

        log.info("Loading bi-encoder  : %s", self.cfg.bi_encoder)
        self._bi_enc = SentenceTransformer(self.cfg.bi_encoder)

        log.info("Loading cross-encoder: %s", self.cfg.cross_encoder)
        self._cross_enc = CrossEncoder(self.cfg.cross_encoder)

    # ── Data Layer ──────────────────────────────────────────────────────────

    def load(self, path: str | Path) -> pd.DataFrame:
        """Supports CSV and Excel input transparently."""
        path = Path(path)
        log.info("Loading candidates from %s", path)
        if path.suffix in (".xlsx", ".xls"):
            df = pd.read_excel(path)
        else:
            df = pd.read_csv(path)
        log.info("  Loaded %d rows, %d columns", *df.shape)
        return df

    def preprocess(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Cleans, validates, and engineers features.
        Gracefully handles missing columns.
        """
        log.info("Preprocessing …")
        df = df.copy()
        df.fillna("", inplace=True)

        # ── Experience ──────────────────────────────────────────────────
        exp_col = self._find_col(df, ["profile_years_of_experience", "years_experience", "experience"])
        if exp_col:
            df["_exp"] = pd.to_numeric(df[exp_col], errors="coerce").fillna(0).clip(lower=0)
        else:
            log.warning("No experience column found – defaulting to 0")
            df["_exp"] = 0.0

        # ── Text fields ─────────────────────────────────────────────────
        title_col   = self._find_col(df, ["profile_current_title",  "current_title", "title"])
        headline_col = self._find_col(df, ["profile_headline", "headline"])
        summary_col  = self._find_col(df, ["profile_summary",  "summary", "bio"])

        def safe(col: Optional[str]) -> pd.Series:
            return df[col].astype(str) if col else pd.Series([""] * len(df))

        # Rich profile: title carries more weight → repeat it
        df["_rich_profile"] = (
            safe(title_col) + ". " + safe(title_col) + ". " +   # title × 2
            safe(headline_col) + ". " +
            safe(summary_col)
        ).str.strip()

        # Raw title for seniority (shorter, cleaner)
        df["_title"] = safe(title_col).str.lower()

        # ── Derived signals ──────────────────────────────────────────────
        df["_skill_score_raw"] = df["_rich_profile"].apply(self._weighted_skill_density)
        df["_seniority_score"] = df["_title"].apply(self._seniority_score)

        # Log-scale + normalise experience
        log_exp = np.log1p(df["_exp"].values)
        df["_exp_norm"] = self._normalise(log_exp)

        # Normalise skill density
        df["_skill_norm"] = self._normalise(df["_skill_score_raw"].values)

        log.info("  Preprocessing complete.")
        return df

    # ── Scoring ─────────────────────────────────────────────────────────────

    def rank(self, df: pd.DataFrame, jd: str) -> list[ScoredCandidate]:
        """Full pipeline: bi-encode → top-K rerank → composite → sort."""
        t0 = time.perf_counter()

        # Stage 1 – Bi-encoder (all candidates)
        log.info("Stage 1 | Bi-encoder embeddings …")
        jd_emb    = self._bi_enc.encode(jd,                         convert_to_tensor=True, show_progress_bar=False)
        prof_embs = self._bi_enc.encode(df["_rich_profile"].tolist(), convert_to_tensor=True, show_progress_bar=True)
        sem_scores = util.cos_sim(jd_emb, prof_embs)[0].cpu().numpy()  # shape (N,)
        df = df.copy()
        df["_sem_raw"] = sem_scores

        # Stage 2 – Cross-encoder rerank on top-K only
        k = min(self.cfg.rerank_top_k, len(df))
        log.info("Stage 2 | Cross-encoder reranking top-%d …", k)
        top_idx = np.argsort(sem_scores)[::-1][:k]
        pairs   = [(jd, df["_rich_profile"].iloc[i]) for i in top_idx]
        rerank_raw = self._cross_enc.predict(pairs, show_progress_bar=True)

        # Build rerank column (non-top-K gets minimum observed score)
        rerank_col = np.full(len(df), rerank_raw.min())
        for pos, idx in enumerate(top_idx):
            rerank_col[idx] = rerank_raw[pos]
        df["_rerank_raw"] = rerank_col

        # Stage 3 – Normalise all signals to [0, 1]
        df["_sem_norm"]    = self._normalise(df["_sem_raw"].values)
        df["_rerank_norm"] = self._normalise(df["_rerank_raw"].values)

        # Stage 4 – Weighted composite
        cfg = self.cfg
        df["_score"] = (
            df["_sem_norm"]    * cfg.w_semantic  +
            df["_rerank_norm"] * cfg.w_rerank    +
            df["_exp_norm"]    * cfg.w_exp       +
            df["_skill_norm"]  * cfg.w_skill     +
            df["_seniority_score"] * cfg.w_seniority
        )

        df_sorted = df.sort_values("_score", ascending=False).reset_index(drop=True)

        id_col = self._find_col(df, ["candidate_id", "id", "cid"]) or df.columns[0]

        results: list[ScoredCandidate] = []
        for rank_idx, row in df_sorted.iterrows():
            sc = ScoredCandidate(
                candidate_id    = str(row[id_col]),
                rank            = int(rank_idx) + 1,
                score           = round(float(row["_score"]), 4),
                semantic_score  = round(float(row["_sem_norm"]), 4),
                rerank_score    = round(float(row["_rerank_norm"]), 4),
                exp_score       = round(float(row["_exp_norm"]), 4),
                skill_score     = round(float(row["_skill_norm"]), 4),
                seniority_score = round(float(row["_seniority_score"]), 4),
                reasoning       = self._explain(row),
            )
            results.append(sc)

        elapsed = time.perf_counter() - t0
        log.info("Ranking complete in %.1fs  |  Top candidate score: %.4f", elapsed, results[0].score)
        return results

    # ── Explanation (XAI) ────────────────────────────────────────────────────

    def _explain(self, row: pd.Series) -> str:
        """
        Produces a structured, human-readable score breakdown.
        No random values — every number is derived from the data.
        """
        title   = str(row.get("_title", "")).title() or "Candidate"
        exp     = float(row.get("_exp", 0))
        skills  = float(row.get("_skill_score_raw", 0))
        sem_pct = round(float(row.get("_sem_norm", 0)) * 100, 1)
        rnk_pct = round(float(row.get("_rerank_norm", 0)) * 100, 1)
        snr_pct = round(float(row.get("_seniority_score", 0)) * 100, 1)

        return (
            f"{title} | {exp:.0f} yrs exp | "
            f"Semantic fit {sem_pct}% | Rerank relevance {rnk_pct}% | "
            f"Weighted skill density {skills:.1f} | Seniority signal {snr_pct}%"
        )

    # ── Signal Extractors ────────────────────────────────────────────────────

    def _weighted_skill_density(self, text: str) -> float:
        """Returns sum of skill weights for all matching skills in text."""
        text_lower = str(text).lower()
        return sum(
            weight
            for skill, weight in self.cfg.skill_taxonomy.items()
            if re.search(r"\b" + re.escape(skill) + r"\b", text_lower)
        )

    def _seniority_score(self, title_lower: str) -> float:
        """Returns highest matching seniority weight in title, or default 0.5."""
        for kw, score in self.cfg.seniority_map.items():
            if kw in title_lower:
                return score
        return 0.50  # assume mid-level if unknown

    # ── Utilities ────────────────────────────────────────────────────────────

    @staticmethod
    def _normalise(arr: np.ndarray) -> np.ndarray:
        """Min-max normalise to [0, 1]; returns zeros if all values identical."""
        mn, mx = arr.min(), arr.max()
        if mx - mn < 1e-9:
            return np.zeros_like(arr, dtype=float)
        return (arr - mn) / (mx - mn)

    @staticmethod
    def _find_col(df: pd.DataFrame, candidates: list[str]) -> Optional[str]:
        """Case-insensitive column lookup; returns first match or None."""
        lower_map = {c.lower(): c for c in df.columns}
        for c in candidates:
            if c.lower() in lower_map:
                return lower_map[c.lower()]
        return None


# ─────────────────────────────────────────────
# Output helpers
# ─────────────────────────────────────────────

def save_submission(results: list[ScoredCandidate], path: str | Path, verbose: bool = True) -> None:
    """Writes competition-format CSV + optional detailed breakdown CSV."""
    path = Path(path)
    fields = ["candidate_id", "rank", "score", "reasoning"]
    with path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        for r in results:
            writer.writerow({k: getattr(r, k) for k in fields})
    log.info("Submission saved → %s", path)

    if verbose:
        detail_path = path.with_stem(path.stem + "_detailed")
        detail_fields = [
            "candidate_id", "rank", "score",
            "semantic_score", "rerank_score", "exp_score",
            "skill_score", "seniority_score", "reasoning",
        ]
        with detail_path.open("w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=detail_fields)
            writer.writeheader()
            for r in results:
                writer.writerow({k: getattr(r, k) for k in detail_fields})
        log.info("Detailed breakdown  → %s", detail_path)


def print_leaderboard(results: list[ScoredCandidate], top_n: int = 10) -> None:
    """Pretty-prints the top-N ranked candidates."""
    separator = "─" * 90
    print(f"\n{'CANDIDATE LEADERBOARD':^90}")
    print(separator)
    print(f"{'RNK':>4}  {'ID':<20}  {'SCORE':>6}  {'SEM':>5}  {'RNKR':>5}  {'EXP':>5}  {'SKILL':>5}  {'SNR':>5}")
    print(separator)
    for r in results[:top_n]:
        print(
            f"{r.rank:>4}  {r.candidate_id:<20}  {r.score:>6.4f}  "
            f"{r.semantic_score:>5.3f}  {r.rerank_score:>5.3f}  "
            f"{r.exp_score:>5.3f}  {r.skill_score:>5.3f}  {r.seniority_score:>5.3f}"
        )
    print(separator)
    print(f"  Showing top {min(top_n, len(results))} of {len(results)} candidates.\n")


# ─────────────────────────────────────────────
# Entry Point
# ─────────────────────────────────────────────

if __name__ == "__main__":
    CANDIDATES_FILE = "/content/sample_candidates.xlsx"
    OUTPUT_FILE     = "/content/final_submission.csv"

    JOB_DESCRIPTION = """
    Seeking a Senior Data / Backend Engineer with strong Python, SQL, and Apache Spark skills.
    The role involves designing and operating large-scale data pipelines and transitioning the
    team toward Machine Learning and Generative AI (GenAI / LLM) infrastructure.
    Ideal candidates have hands-on experience with MLOps tooling (MLflow, Airflow, dbt),
    cloud-native architectures, and a track record of delivering production ML systems.
    """

    # ── Optional: override config ──────────────────────────────────────────
    config = EngineConfig(
        w_semantic  = 0.50,
        w_rerank    = 0.20,
        w_exp       = 0.15,
        w_skill     = 0.10,
        w_seniority = 0.05,
        rerank_top_k = 50,
    )

    # ── Run ────────────────────────────────────────────────────────────────
    engine = CandidateDiscoveryEngine(config)

    df          = engine.load(CANDIDATES_FILE)
    df          = engine.preprocess(df)
    results     = engine.rank(df, JOB_DESCRIPTION)

    print_leaderboard(results, top_n=10)
    save_submission(results, OUTPUT_FILE, verbose=True)

    log.info("Done. 🎯")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]


                                  CANDIDATE LEADERBOARD                                   
──────────────────────────────────────────────────────────────────────────────────────────
 RNK  ID                     SCORE    SEM   RNKR    EXP  SKILL    SNR
──────────────────────────────────────────────────────────────────────────────────────────
   1  CAND_0000001          0.8822  0.916  1.000  0.663  1.000  0.500
   2  CAND_0000010          0.8697  1.000  0.856  0.491  1.000  0.500
   3  CAND_0000044          0.7373  0.849  0.795  0.580  0.417  0.500
   4  CAND_0000025          0.7008  0.869  0.609  0.688  0.167  0.500
   5  CAND_0000043          0.6992  0.896  0.490  0.744  0.167  0.500
   6  CAND_0000014          0.6985  0.836  0.632  0.750  0.167  0.500
   7  CAND_0000018          0.6840  0.839  0.632  0.643  0.167  0.500
   8  CAND_0000015          0.6625  0.871  0.510  0.557  0.167  0.500
   9  CAND_0000023          0.6607  0.885  0.581  0.403  0.167  0.500
  10  CAND_0000027        